In [2]:
from nlp4bia.datasets.benchmark.distemist import DistemistLoader, DistemistGazetteer
from sentence_transformers import SentenceTransformer
from nlp4bia.linking import BECELinker

# 1) Load data
df = DistemistLoader().df
df_gaz = DistemistGazetteer().df#.iloc[:100]

# biencoder_path = "/gpfs/projects/bsc14/abecerr1/hub/models--ICB-UMA--ClinLinker-KB-GP/snapshots/8f914c58a1cbcff43331eb15b101eaa5e5c6920a"
# biencoder_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/biencoder_medprocner_1_epoch_32_batch_5_parents_stag"
biencoder_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/enfermedad/distemist/biencoder_distemist_1_epoch_32_batch_5_parents_stag"
# ce_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
ce_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/enfermedad/distemist/crossencoder_distemist_5_epoch_16_batch"

# # 2) Prepare a SentenceTransformer bi-encoder (already loaded)
# biencoder_model = SentenceTransformer(biencoder_path)
biencoder_model = SentenceTransformer(biencoder_path, device="cuda")
vector_db = biencoder_model.encode(
    df_gaz["term"].tolist(),
    batch_size=4096,
    show_progress_bar=True,
    convert_to_tensor=True
)


Batches: 100%|██████████| 36/36 [00:12<00:00,  2.90it/s]


In [4]:

# 3) Initialize the BECELinker
linker = BECELinker(
    df_gazetteer=df_gaz,
    biencoder_model_or_path=biencoder_path,
    crossencoder_model_or_path=ce_path,
    biencoder_batch_size=4096,
    reranker_batch_size=4096,
    vector_db=vector_db,
)

# 4) Link a list of mentions
ls_mentions = df["span"].tolist()[:10]
results = linker.link(
    mentions=ls_mentions,
    n_candidates=200,
    top_k=5,
    return_documents=True
)

# 5) Inspect output
for res in results:
    print(f"Mention: {res['mention']}")
    for idx, (term, code, score) in enumerate(zip(res["terms"], res["codes"], res["similarity"]), start=1):
        print(f"  {idx:02d}. {term} ({code}) → {score:.4f}")
    print()


Initializing DenseRetriever...
Using bi-encoder model: /gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/enfermedad/distemist/biencoder_distemist_1_epoch_32_batch_5_parents_stag
Note: Vector DB will be computed on the fly. Increase `biencoder_batch_size` to accelerate this.
In case of MemoryError, try reducing `biencoder_batch_size` or using a smaller model.
DenseRetriever initialized successfully.
Initializing CrossEncoder Reranker...
Using CrossEncoder model: /gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/enfermedad/distemist/crossencoder_distemist_5_epoch_16_batch


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.52it/s]


Mention: DM
  01. diabetes sacarina (73211009) → 0.9858
  02. diabetes mellitus (73211009) → 0.9854
  03. [X]otra diabetes mellitus especificada (191045007) → 0.0192
  04. DMG (11687002) → 0.0111
  05. diabetes mellitus insulino - dependiente (46635009) → 0.0026

Mention: deshidratación
  01. deshidratación (34095006) → 0.9869
  02. deshidratación, no clasificada en otra parte (190896001) → 0.9712
  03. pérdida de agua (89599006) → 0.0497
  04. depleción de volumen (28560003) → 0.0180
  05. sobrehidratación (61688009) → 0.0073

Mention: hiperamilasemia
  01. elevación de la amilasa sérica (275739007) → 0.9238
  02. fosfatasa alcalina sérica elevada (166627004) → 0.0681
  03. fosfatasa alcalina elevada (274770006) → 0.0344
  04. deficiencia de sacarógeno amilasa (124453001) → 0.0038
  05. bilirrubinemia (20505009) → 0.0033

Mention: pancreatitis aguda
  01. pancreatitis aguda (197456007) → 0.9866
  02. pancreatitis aguda (39726008) → 0.9866
  03. inflamación aguda del páncreas (39726008

In [ ]:
# Upload model to hub
# linker.retriever.model.push_to_hub("BSC-NLP4BIA/Distemist-Biencoder",
#                                   token="",
#                                   private=False)

model.safetensors: 100%|██████████| 504M/504M [00:16<00:00, 29.7MB/s] 


'https://huggingface.co/BSC-NLP4BIA/Distemist-Biencoder/commit/8e49d943511b0932ac95d7fb90d1b98a6e570fbc'

In [ ]:
# # Upload model to hub
# linker.reranker.model.push_to_hub("BSC-NLP4BIA/Distemist-CE-Reranker",
#                                   token="",
#                                   private=False)

model.safetensors: 100%|██████████| 504M/504M [00:21<00:00, 23.8MB/s] 


'https://huggingface.co/BSC-NLP4BIA/Distemist-CE-Reranker/commit/3ee6d97edf649651ad7d285bad26643f4bf31052'